## 📑 HAPS Modifications Monitoring

#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AYALH)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()

from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

import boto3
import os

#takes the values d for dev, qa for test and p for prod depending in which workspace the init variables script is running
# env = os.environ"S3PREFIX" 

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `p-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

##### Schema of the table

In [0]:
table_name = 'HAPS_ModificationsMonitoring'

schema = (
      "BusinessDate date,"
      "PortfolioName string,"
      "CDI string,"
      "AgreementKey string," 
      "ExposureKey string,"
      "AssetSegment string,"
      "IsActiveFlag boolean,"
      "ApplicationKey string,"
      "ApplicationDateCreated date,"
      "ApplicationDateUpdated date,"
      "ApplicationActivationDate date,"
      "ApplicationStatus string,"
      "ApplicationCancelledDate date,"
      "ApplicationRejectedDate date,"
      "AccountImplementationStatus string,"
      "ImplementationDate date,"
      "NewExposureKey string,"
      "ModificationType string,"
      "CreditType string,"
      "FlowSegment string,"
      "DeclaredAffordability decimal(19,2),"
      "StatedAffordability decimal(19,2),"
      "IncomeUsedForCalculations decimal(19,2),"
      "CurrentStepName string,"
      "ParentStepGroupingDescription string,"
      "CommunicationChannel string,"
      "UserCreatedChannel string,"
      "UserCreated string,"
      "Channel string,"
      "SoMApplicationActivationDate date,"
      "SoMImplementationDate date,"
      "ApplicationStatusCategory string,"
      "DeclaredAffordabilityGroup string,"
      "StatedAffordabilityGroup string,"
      "IncomeUsedForCalculationsGroup string,"
      "FLDTIncomeSource string,"
      "SoMAppDateArrearsStatusENG string,"
      "SoMAppDateSourceBucket int,"
      "SoMAppDateGBVEUR decimal(19,2),"
      "SoMAppDateOnBalanceAmountEUR decimal(19,2),"
      "SoMAppDateObjectiveDate date,"
      "SoMAppDateObjectiveStatus string,"
      "SoMImplDateArrearsStatusENG string,"
      "SoMImplDateSourceBucket int,"
      "SoMImplDateGBVEur decimal(19,2),"
      "SoMImplDateOnBalanceAmountEur decimal(19,2),"
      "SoMImplDateObjectiveDate date,"
      "SoMImplSoMDateObjectiveStatus string,"
      "ArrearsStatusENG string,"
      "SourceBucket int,"
      "GBVEur decimal(19,2),"
      "OnBlanceAmountEur decimal(19,2),"
      "DelinquentEur decimal(19,2),"
      "GBVAfterModificationEur decimal(19,2),"
      "ImplementedTotalPaymentsEUR decimal(19,2),"
      "ImplementedInstallmentAmount decimal(19,2),"
      "ImplementedInstallmentAmountGroup string,"
      "ImplementedDueInstallmentBucket int,"
      "DaysToImplementation int,"
      "DaysToCancellation int,"
      "DaysToRejection int,"
      "DaysActive int,"
      "ImplementedPaymentcategory string,"
      "InsertTimeStamp timestamp"
)

In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date,  # type: ignore
)


##### LastWorkingDates

In [0]:
LastWorkingDates = spark.sql(
f'''
SELECT CalendarYear,
CalendarMonth,
MAX(CalendarDate) AS LastWorkingDate
FROM  `{env_prefix}-cdh`.reference.ref_calendar
WHERE IsBusinessDay = 1
GROUP BY CalendarYear, CalendarMonth
'''
)
LastWorkingDates.createOrReplaceTempView("LastWorkingDates")

##### haps_portfolios

In [0]:
haps_portfolios = spark.sql(
f'''
SELECT DISTINCT A.PortfolioName
FROM `{env_prefix}-reportinputfiles`.repoint.haps_repointportfolios  A
INNER JOIN (SELECT BusinessDate, MAX(IngestionTimeStamp) IngestionTimeStamp, PortfolioName 
            FROM `{env_prefix}-reportinputfiles`.repoint.haps_repointportfolios
            GROUP BY  BusinessDate, PortfolioName) B
ON A.BusinessDate = B.BusinessDate AND A.PortfolioName = B.PortfolioName AND A.IngestionTimeStamp = B.IngestionTimeStamp
WHERE A.Report = 'HAPS_ModificationsMonitoring'
AND A.IsActive = 1 
''' 
)
haps_portfolios.createOrReplaceTempView("haps_portfolios")

##### portfolio

In [0]:
portfolio = spark.sql(
f'''
SELECT distinct P.BusinessDate,  lower(P.ParentPortfolio) as ParentPortfolio, P.AssignmentDate, P.PortfolioName
FROM `{env_prefix}-cdh`.primary.portfolio P
INNER JOIN haps_portfolios HP ON P.PortfolioName = HP.PortfolioName
WHERE P.BusinessDate = {run_date!r}
'''
)
portfolio.createOrReplaceTempView("portfolio")

##### fldt_allApplications

In [0]:
FLDT_AllApplications = spark.sql(
f'''
SELECT app.*, cast(AssignmentDate as date) as AssignmentDate
FROM `{env_prefix}-cdh-delivery`.repoint.HAPS_FLDTAllApplications app
left join portfolio p on lower(p.parentPortfolio) = lower(app.customer_portfolioName) 
WHERE 1=1 
AND app.BusinessDate = {run_date!r} 
AND ApplicationActivationDate <= {run_date!r}
AND ActivatedApplicationFlag = 1
'''
)
FLDT_AllApplications.createOrReplaceTempView("FLDT_AllApplications")

##### RefDates

In [0]:
RefDates = spark.sql(
f'''
SELECT DISTINCT SoMApplicationActivationDate RefDate 
FROM   
(
    SELECT 		
    	CASE WHEN last_day(add_months(CAST(app.ApplicationActivationDate AS DATE), -1))  < app.AssignmentDate	THEN app.AssignmentDate 
						 ELSE lwd.LastWorkingDate 
					END		AS SoMApplicationActivationDate 
    FROM FLDT_AllApplications app
    LEFT JOIN LastWorkingDates lwd 
    ON lwd.CalendarYear = year(last_day(add_months(CAST(app.ApplicationActivationDate AS DATE), -1))) 
    AND lwd.CalendarMonth = month(last_day(add_months(CAST(app.ApplicationActivationDate AS DATE), -1)))
)
UNION
SELECT DISTINCT SoMImplementationDate RefDate
FROM (
    SELECT 		CASE WHEN last_day(add_months(CAST(app.QualcoAppDisbursedDate AS DATE), -1))  < app.AssignmentDate	THEN app.AssignmentDate  
                ELSE lwd2.LastWorkingDate 
        END	AS SoMImplementationDate
    FROM FLDT_AllApplications app
    LEFT JOIN LastWorkingDates lwd2 
    ON lwd2.CalendarYear = year(last_day(add_months(CAST(app.QualcoAppDisbursedDate AS DATE), -1))) 
    AND lwd2.CalendarMonth = month(last_day(add_months(CAST(app.QualcoAppDisbursedDate AS DATE), -1)))
)
UNION
SELECT cast({run_date!r} as date)
'''
)
RefDates.createOrReplaceTempView("RefDates")


##### Modifications

In [0]:
Modifications = spark.sql(
f'''
SELECT DISTINCT 
NewExposureKey,
restructuringDate AS RestructuringDate
FROM `{env_prefix}-cdh`.primary.restructuring
WHERE RestructuringDate <= {run_date!r}

'''
)
Modifications.createOrReplaceTempView("modifications")

##### dexposure

In [0]:
dexposure = spark.sql(f'''
        select d.* 
        from `{env_prefix}-cdh`.derivative.dexposure d           
        inner join refDates r on r.RefDate = d.BusinessDate   
''')
dexposure.createOrReplaceTempView("dexposure")

##### SourceBucket

In [0]:
from pyspark.sql.types import IntegerType
from pyspark.sql.types import StringType
from pyspark.sql.types import BooleanType
from pyspark.sql.functions import udf

def get_bucket_from_dpd( dpd : int ):
        
    if dpd is None:
        return 0 # None
    elif dpd <= 0:
        return 0
    elif 1 <= dpd <= 29:
        return 1
    elif 30 <= dpd <= 59:
        return 2
    elif 60 <= dpd <= 89:
        return 3
    elif 90 <= dpd <= 119:
        return 4
    elif 120 <= dpd <= 149:
        return 5
    elif 150 <= dpd <= 179:
        return 6
    elif 180 <= dpd <= 209:
        return 7
    elif 210 <= dpd <= 239:
        return 8
    elif 240 <= dpd <= 269:
        return 9
    elif 270 <= dpd <= 299:
        return 10
    elif 300 <= dpd <= 329:
        return 11
    elif 330 <= dpd <= 359:
        return 12
    elif 360 <= dpd <= 720:
        return 13
    elif dpd > 720:
        return 14


# Convert your Python function into a UDF
get_bucket_from_dpd_udf = udf(get_bucket_from_dpd, IntegerType())



def is_valid_tax_number(Input : str):
    if Input is None:
        return None
    elif not Input.isdigit():
        return None
    elif Input == '999999993':
        return None
    elif any(c in Input for c in ['+', '-', '.', ',', 'E', 'e']):
        return None 
    elif len(Input) == 8:
        Input = '0' + Input
    if len(Input) != 9:
        return None
    factor = 256
    check_digit = int(Input[8])
    value =0
    for i in range(0,8):
        digit = int(Input[i])
        value = value + digit * factor
        factor = factor // 2
    if value == 0:
        return False
    checksum = value % 11
    if checksum == 10:
        checksum=0
    if checksum == check_digit:
        return True
    else:
        return False

# Convert your Python function into a UDF
is_valid_tax_number_udf = udf(is_valid_tax_number, BooleanType())
dexposure = dexposure.withColumn("SourceBucket", get_bucket_from_dpd_udf(dexposure["DaysPastDueWithMateriality"]))
dexposure.createOrReplaceTempView("dexposure")

In [0]:
dexposure_active  = spark.sql(
f'''
SELECT  
    A.BusinessDate
    , A.PortfolioName
    , A.ExposureKey
    , A.AssetSegment
    , A.AgreementKey
    , A.IsActiveFlag
    , `{env_prefix}-cdh`.derivative.converttoeur(A.Currency, A.OnBalance, A.BusinessDate) OnBalance
    , `{env_prefix}-cdh`.derivative.converttoeur(A.Currency, A.GBV, A.BusinessDate) GBV
    , A.ObjectiveStatus
    , A.ObjectiveStatusdate
    , A.SourceBucket AS SourceBucket
    , A.ExposureStatus as ArrearsStatusENG
    , `{env_prefix}-cdh`.derivative.converttoeur(A.Currency, A.DueAmount, A.BusinessDate) as DueAmount
    , M.GBVAtModificationDate
    , M.NewArrearsStatusENG
    , M.NewSourceBucket
FROM dexposure A
inner join Portfolio p 
on p.PortfolioName = a.PortfolioName
LEFT JOIN (
select ExposureKey, BusinessDate
    , rank() over (partition by e.ExposureKey order by M.RestructuringDate) as Rank
    from `{env_prefix}-cdh`.derivative.dexposure e
    inner join Modifications M
    on e.ExposureKey = M.NEWExposureKey 
    and e.BusinessDate = M.RestructuringDate
    where e.SourceSystem = 'profits'
        and e.BusinessDate <= {run_date!r}
) B
on A.ExposureKey = B.ExposureKey and B.Rank = 1
LEFT JOIN 
    (
    select M.NewExposureKey, M.RestructuringDate, 
    e.BusinessDate, e.ExposureStatus as NewArrearsStatusENG, e.SourceBucket as NewSourceBucket,
    `{env_prefix}-cdh`.derivative.converttoeur(e.Currency, e.GBV, e.BusinessDate) as GBVAtModificationDate
    FROM Modifications M 
    INNER JOIN dexposure e
    on e.ExposureKey = M.NEWExposureKey 
    and e.BusinessDate = M.RestructuringDate
    ) M
ON B.ExposureKey = M.NewExposureKey AND B.BusinessDate = M.RestructuringDate AND B.Rank = 1
WHERE A.IsActiveFlag = true
'''
)
dexposure_active.createOrReplaceTempView("dexposure_active")

In [0]:
dexposure_inactive = spark.sql(
f'''
SELECT 
    {run_date!r} as BusinessDate
    , A.PortfolioName
    , A.ExposureKey
    , A.AssetSegment
    , A.AgreementKey
    ,  false  as IsActiveFlag
    , `{env_prefix}-cdh`.derivative.converttoeur(A.Currency, A.OnBalance, A.BusinessDate) OnBalance
    , `{env_prefix}-cdh`.derivative.converttoeur(A.Currency, A.GBV, A.BusinessDate) GBV
    , A.ObjectiveStatus
    , A.ObjectiveStatusdate
    , case when dayspastduewithmateriality is not null then A.SourceBucket else null end as SourceBucket
    , A.ExposureStatus as ArrearsStatusENG
    , `{env_prefix}-cdh`.derivative.converttoeur(A.Currency, A.DueAmount, A.BusinessDate) as DueAmount   
    , M.GBVAtModificationDate
    , M.NewArrearsStatusENG
    , M.NewSourceBucket
from (select e.*
      , rank() over (partition by e.ExposureKey order by e.BusinessDate desc) as Rank
      from dexposure e
      inner join Portfolio p 
      on p.PortfolioName = e.PortfolioName
      where e.SourceSystem = 'profits'
        and e.IsActiveFlag = false
        and e.BusinessDate <= {run_date!r}
        ) AS A
LEFT JOIN (
select ExposureKey, BusinessDate
    , rank() over (partition by e.ExposureKey order by M.RestructuringDate) as Rank
    from `{env_prefix}-cdh`.derivative.dexposure e
    inner join Modifications M
    on e.ExposureKey = M.NEWExposureKey 
    and e.BusinessDate = M.RestructuringDate
    where e.SourceSystem = 'profits'
        and e.BusinessDate <= {run_date!r}
) B
on A.ExposureKey = B.ExposureKey and B.Rank = 1
LEFT JOIN 
    (
    select M.NewExposureKey, M.RestructuringDate, 
    e.BusinessDate, e.ExposureStatus as NewArrearsStatusENG, e.SourceBucket as NewSourceBucket,
    `{env_prefix}-cdh`.derivative.converttoeur(e.Currency, e.GBV, e.BusinessDate) as GBVAtModificationDate
    FROM Modifications M 
    INNER JOIN dexposure e
    on e.ExposureKey = M.NEWExposureKey 
    and e.BusinessDate = M.RestructuringDate
    ) M
ON B.ExposureKey = M.NewExposureKey AND B.BusinessDate = M.RestructuringDate AND B.Rank = 1
left anti join dexposure_active e_pf_act on e_pf_act.ExposureKey = A.ExposureKey and e_pf_act.BusinessDate = {run_date!r}
where A.Rank = 1
'''
)
dexposure_inactive.createOrReplaceTempView("dexposure_inactive")

In [0]:
Dexposure = spark.sql(
f'''
select *
from dexposure_active

union all

select *
from dexposure_inactive

'''
)
Dexposure.createOrReplaceTempView("dexposure")

##### Account

In [0]:
Account = spark.sql(
f'''
select d.*
from dexposure d
-- inner join refDates r on r.RefDate = d.BusinessDate
'''
)
Account.createOrReplaceTempView("Account")

##### Applications

In [0]:
Applications = spark.sql(
f'''
SELECT * FROM (
		SELECT   DISTINCT 
					acc.PortfolioName																				 AS PortfolioName	
				,	acc.AgreementKey
				,  app.unifiedLoan_AccountNumberWithPrefix													     AS ExposureKey 
				,	app.customer_CustomerCode																	 AS CDI	
				,	acc.AssetSegment
				,	acc.IsActiveFlag
				,	app.app_ApplicationKey																	     AS ApplicationKey				
				,	CAST(app.app_DateCreated AS DATE)															 AS ApplicationDateCreated
				,	CAST(app.app_CurrentStepAssignmentDate AS DATE)												 AS ApplicationDateUpdated
				,	CAST(app.ApplicationActivationDate AS DATE)													 AS ApplicationActivationDate	
				,	app.ApplicationImplementationStatus															 AS ApplicationStatus	
				
				,	CASE WHEN app.ApplicationImplementationStatus = 'Cancelled' THEN CAST(app.app_CurrentStepAssignmentDate AS DATE)															
						 ELSE NULL END																				 AS ApplicationCancelledDate
				,	CAST(app.QualcoAppRejectedDate AS DATE)																	 AS ApplicationRejectedDate
				,	app.AccountImplementationStatus																		
				,	CAST(app.QualcoAppDisbursedDate AS DATE)													AS ImplementationDate 
				,   CASE WHEN app.profitsInfo_NewAccountNumber IS NULL THEN app.profitsInfo_NewAccountNumber
						 ELSE CONCAT(LEFT(app.unifiedLoan_AccountNumberWithPrefix,3),app.profitsInfo_NewAccountNumber) 
					END																								 AS NewAccountNumber
				,	app.product_ProductCategoryType																 AS ModificationType
				,	app.product_AutoCreditStatusDescription													     AS CreditType
				,	app.app_FlowSegmentDescription															     AS FlowSegment
				,	app.customer_DeclaredAffordability															 AS DeclaredAffordability
				,	app.customer_StatedAffordability															 AS StatedAffordability
				,	app.customer_IncomeUsedForCalculations														 AS IncomeUsedForCalculations
				,	app.app_CurrentStepIntegrationCode															 AS CurrentStepName
				,	app.galaxyApplicationMetadata_ParentStepGroupingDescription									 AS ParentStepGroupingDescription
				,	app.app_CommunicationChannelDescription													     AS CommunicationChannel			
				,	app.app_UserCreatedGroupName																 AS UserCreatedChannel			
				,	app.app_UserCreated																			 AS UserCreated	
				,	app.unifiedLoan_DecisionTreeValidPath  
				,	CASE WHEN last_day(add_months(CAST(app.ApplicationActivationDate AS DATE), -1))  < app.AssignmentDate	THEN app.AssignmentDate
						 ELSE lwd.LastWorkingDate 
					END		AS SoMApplicationActivationDate 
				,	CASE WHEN last_day(add_months(CAST(app.QualcoAppDisbursedDate AS DATE), -1))  < app.AssignmentDate	THEN app.AssignmentDate 
						 ELSE lwd2.LastWorkingDate 
					END																								 AS SoMImplementationDate 
				,	CASE WHEN app.AccountImplementationStatus     = 'Implemented'																		THEN 'Implemented'
						 WHEN app.ApplicationImplementationStatus = 'Cancelled' AND app.QualcoAppRejectedDate						    IS NOT NULL THEN 'Rejected'
						 WHEN app.ApplicationImplementationStatus = 'Cancelled'																			THEN 'Canceled'
						 WHEN app.ApplicationImplementationStatus = 'Active'																			THEN 'Active'
																																							ELSE NULL
					END																								 AS ApplicationStatusCategory 
				,	RANK() OVER (PARTITION BY app.unifiedLoan_AccountNumberWithPrefix
									 ORDER BY CASE WHEN app.AccountImplementationStatus		 = 'Implemented' THEN 1
												   WHEN app.ApplicationImplementationStatus  = 'Active'		 THEN 2
																												 ELSE 3
											   END ASC,
											   app.ApplicationActivationDate DESC,
											   app.app_CurrentStepAssignmentDate DESC,
											   app.app_UserCreatedGroupName ASC
											   )						AS Rank  
FROM FLDT_AllApplications app
LEFT JOIN LastWorkingDates lwd 
ON lwd.CalendarYear = year(last_day(add_months(CAST(app.ApplicationActivationDate AS DATE), -1))) 
AND lwd.CalendarMonth = month(last_day(add_months(CAST(app.ApplicationActivationDate AS DATE), -1)))
LEFT JOIN LastWorkingDates lwd2 
ON lwd2.CalendarYear = year(last_day(add_months(CAST(app.QualcoAppDisbursedDate AS DATE), -1))) 
AND lwd2.CalendarMonth = month(last_day(add_months(CAST(app.QualcoAppDisbursedDate AS DATE), -1)))
LEFT JOIN Account acc
ON acc.ExposureKey = app.unifiedLoan_AccountNumberWithPrefix  and acc.BusinessDate = {run_date!r} 
WHERE 1=1 
)
WHERE Rank = 1
AND ApplicationStatusCategory IS NOT NULL
'''
)
Applications.createOrReplaceTempView("Applications")

##### payments

In [0]:

Payments = spark.sql(
f'''
select ExposureKey, PaymentAmount, PaymentAmountEUR, PaymentActualDate, PaymentType from (
    SELECT p.ExposureKey, PaymentAmount, `{env_prefix}-cdh`.derivative.converttoeur(Currency, PaymentAmount, PaymentActualDate) as PaymentAmountEUR, PaymentActualDate, PaymentType, rank() over (partition by PaymentKey order by p.BusinessDate desc) as rank
    FROM  `{env_prefix}-cdh`.primary.payment p
    inner join Portfolio hp 
    on hp.PortfolioName = p.PortfolioName
    WHERE 1=1 
    AND PaymentActualDate <= {run_date!r}
    AND PaymentType = 'Είσπραξη από πελάτη' 
) 
where rank = 1
'''
)
Payments.createOrReplaceTempView("payments")


In [0]:
PaymentsAgg = spark.sql(
'''
SELECT 
ExposureKey,
SUM(PaymentAmount) AS PaymentAmount,
SUM(PaymentAmountEUR) as PaymentAmountEUR
FROM payments p 
INNER JOIN (SELECT DISTINCT NewAccountNumber FROM applications) app
ON p.ExposureKey = app.NewAccountNumber
GROUP BY ExposureKey
'''
)
PaymentsAgg.createOrReplaceTempView("PaymentsAgg")

##### paymentSchedule

In [0]:
PaymentSchedule = spark.sql(
f'''
SELECT Exposurekey, InstallmentAmount, InstallmentDate 
FROM (
        SELECT ExposureKey, abs(InstallmentAmount) as InstallmentAmount, cast(InstallmentDate as Date), rank() over (partition by ExposureKey order by ps.BusinessDate desc) as Rank
        FROM `{env_prefix}-cdh`.primary.paymentschedule ps 
        inner join Portfolio p 
        on p.PortfolioName = ps.PortfolioName
        WHERE 1=1 
        AND InstallmentAmount <> 0 
        AND InstallmentDate <= {run_date!r}
        AND ps.BusinessDate <= {run_date!r}  
        AND ifnull(ImplementationFlag, 'true') = 'true'
        AND InstallmentType = 'Installment'
) WHERE RANK = 1
'''
)
PaymentSchedule.createOrReplaceTempView("PaymentSchedule")

##### FirstInstallment

In [0]:
FirstInstallment = spark.sql(
f''' 
select ps1.ExposureKey, sum(ps1.InstallmentAmount) as InstallmentAmount
from paymentschedule ps1
inner join 
(
    SELECT ps.ExposureKey, 
    MIN(ps.InstallmentDate) AS MinInstallmentDate 
    FROM  PaymentSchedule ps
    INNER JOIN (SELECT DISTINCT NewAccountNumber FROM applications) app
    ON ps.ExposureKey = app.NewAccountNumber
    GROUP BY ps.ExposureKey
) ps2
on ps1.ExposureKey = ps2.ExposureKey and ps1.InstallmentDate = ps2.MinInstallmentDate
group by ps1.ExposureKey
'''
)
FirstInstallment.createOrReplaceTempView("FirstInstallment")

#### FLDTChannel

In [0]:
# %python
# create_view_sql = f"""
# CREATE OR REPLACE VIEW `qa-cdh-delivery`.repoint.haps_fldtchannel
# WITH SCHEMA COMPENSATION
# AS
# SELECT 'COSMOS TEAM ATTICA'																	AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'COSMOS TEAM RURAL'																	AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'Credit Workout Decision Maker-Non L.3869 - Segment >€100k Assistant Manager'		AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'Credit Workout Decision Maker-Non L.3869 - Segment >€100k Officer/Senior Officer'	AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'Credit Workout Decision Maker-Non L.3869 - Segment >€50k Officer/Senior Officer'	AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'Credit Workout Processor-Non L.3869 - Segment >€100k'								AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'Credit Workout Processor-Non L.3869 - Segment >€50k'								AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'FAs'																				AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'GALAXY TEAM 1'																		AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'GALAXY TEAM 2'																		AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'GALAXY TEAM 3'																		AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'GALAXY TEAM 4'																		AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'GALAXY TEAM 5'																		AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'Hubs Management'																	AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'ORION TEAM'																			AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'Pilot FAs'																			AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'ΠΕΡ. ΚΕΝΤΡΟ ΔΙΑΧ/ΣΕΩΣ ΚΑΘ/ΣΕΩΝ-XANIA'												AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'ΠΕΡ. ΚΕΝΤΡΟ ΔΙΑΧ/ΣΕΩΣ ΚΑΘ/ΣΕΩΝ-ΑΝΑΤ. ΑΤΤΙΚΗ'										AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'ΠΕΡ. ΚΕΝΤΡΟ ΔΙΑΧ/ΣΕΩΣ ΚΑΘ/ΣΕΩΝ-ΑΝΑΤ. ΘΕΣ/ΝΙΚΗ'										AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'ΠΕΡ. ΚΕΝΤΡΟ ΔΙΑΧ/ΣΕΩΣ ΚΑΘ/ΣΕΩΝ-ΒΟΛΟΣ'												AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'ΠΕΡ. ΚΕΝΤΡΟ ΔΙΑΧ/ΣΕΩΣ ΚΑΘ/ΣΕΩΝ-ΒΟΡΕΙΑ ΑΤΤΙΚΗ'										AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'ΠΕΡ. ΚΕΝΤΡΟ ΔΙΑΧ/ΣΕΩΣ ΚΑΘ/ΣΕΩΝ-ΒΟΡΕΙΟΔΥΤΙΚΗ ΑΤΤΙΚΗ'									AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'ΠΕΡ. ΚΕΝΤΡΟ ΔΙΑΧ/ΣΕΩΣ ΚΑΘ/ΣΕΩΝ-ΔΥΤΙΚΗ ΑΤΤΙΚΗ'										AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'ΠΕΡ. ΚΕΝΤΡΟ ΔΙΑΧ/ΣΕΩΣ ΚΑΘ/ΣΕΩΝ-ΔΥΤΙΚΗ ΘΕΣ/ΝΙΚΗ'										AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'ΠΕΡ. ΚΕΝΤΡΟ ΔΙΑΧ/ΣΕΩΣ ΚΑΘ/ΣΕΩΝ-ΗΡΑΚΛΕΙΟ'											AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'ΠΕΡ. ΚΕΝΤΡΟ ΔΙΑΧ/ΣΕΩΣ ΚΑΘ/ΣΕΩΝ-ΙΩΑΝΝΙΝΑ'											AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'ΠΕΡ. ΚΕΝΤΡΟ ΔΙΑΧ/ΣΕΩΣ ΚΑΘ/ΣΕΩΝ-ΚΑΒΑΛΑ'												AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'ΠΕΡ. ΚΕΝΤΡΟ ΔΙΑΧ/ΣΕΩΣ ΚΑΘ/ΣΕΩΝ-ΚΕΝΤΡΟ ΑΘΗΝΩΝ'										AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'ΠΕΡ. ΚΕΝΤΡΟ ΔΙΑΧ/ΣΕΩΣ ΚΑΘ/ΣΕΩΝ-ΚΟΖΑΝΗ'												AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'ΠΕΡ. ΚΕΝΤΡΟ ΔΙΑΧ/ΣΕΩΣ ΚΑΘ/ΣΕΩΝ-ΚΟΡΙΝΘΟΣ'											AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'ΠΕΡ. ΚΕΝΤΡΟ ΔΙΑΧ/ΣΕΩΣ ΚΑΘ/ΣΕΩΝ-Λ.ΒΟΥΛΙΑΓΜΕΝΗΣ'										AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'ΠΕΡ. ΚΕΝΤΡΟ ΔΙΑΧ/ΣΕΩΣ ΚΑΘ/ΣΕΩΝ-Λ.ΣΥΓΓΡΟΥ'											AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'ΠΕΡ. ΚΕΝΤΡΟ ΔΙΑΧ/ΣΕΩΣ ΚΑΘ/ΣΕΩΝ-ΛΑΡΙΣΑ'												AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'ΠΕΡ. ΚΕΝΤΡΟ ΔΙΑΧ/ΣΕΩΣ ΚΑΘ/ΣΕΩΝ-ΠΑΓΚΡΑΤΙ'											AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'ΠΕΡ. ΚΕΝΤΡΟ ΔΙΑΧ/ΣΕΩΣ ΚΑΘ/ΣΕΩΝ-ΠΑΤΡΑ'												AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'ΠΕΡ. ΚΕΝΤΡΟ ΔΙΑΧ/ΣΕΩΣ ΚΑΘ/ΣΕΩΝ-ΠΕΙΡΑΙΑΣ'											AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'ΠΕΡ. ΚΕΝΤΡΟ ΔΙΑΧ/ΣΕΩΣ ΚΑΘ/ΣΕΩΝ-ΠΟΛΗ ΘΕΣ/ΝΙΚΗΣ'										AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'ΠΕΡ. ΚΕΝΤΡΟ ΔΙΑΧ/ΣΕΩΣ ΚΑΘ/ΣΕΩΝ-ΡΟΔΟΣ'												AS UserCreatedChannel, 'FAs'	Channel UNION
# SELECT 'Call Center IRT'																	AS UserCreatedChannel, 'IRT'	Channel UNION
# SELECT 'Credit Workout Decision Maker-Non L.3869 - Segment <€100k Officer/Senior Officer'	AS UserCreatedChannel, 'IRT'	Channel UNION
# SELECT 'Credit Workout Decision Maker-Non L.3869 - Segment <€50k Officer/Senior Officer'	AS UserCreatedChannel, 'IRT'	Channel UNION
# SELECT 'IRT'																				AS UserCreatedChannel, 'IRT'	Channel UNION
# SELECT 'Pilot IRT'																			AS UserCreatedChannel, 'IRT'	Channel UNION
# SELECT 'Operations Validator'																AS UserCreatedChannel, 'N/A'	Channel UNION
# SELECT 'Super Portfolio Group'																AS UserCreatedChannel, 'N/A'	Channel """
# spark.sql(create_view_sql)

In [0]:
VW_FLDTChannel = spark.sql(
f'''
select * from `{env_prefix}-cdh-delivery`.repoint.haps_fldtchannel
'''
)
VW_FLDTChannel.createOrReplaceTempView('VW_FLDTChannel')

#### final_df

In [0]:
ModificationsMonitoring = spark.sql(
f'''
		SELECT 
					App.PortfolioName
				,	App.CDI
				,	App.AgreementKey
				,	App.ExposureKey
				,	App.AssetSegment
				,	App.IsActiveFlag
				,	App.ApplicationKey
				,	App.ApplicationDateCreated							
				,	App.ApplicationDateUpdated							
				,	App.ApplicationActivationDate						
				,	App.ApplicationStatus						
				,	App.ApplicationCancelledDate							
				,	App.ApplicationRejectedDate							
				,	App.AccountImplementationStatus				
				,	App.ImplementationDate							
				,	App.NewAccountNumber							
				,	App.ModificationType							
				,	App.CreditType								
				,	App.FlowSegment								
				,	App.DeclaredAffordability					
				,	App.StatedAffordability					
				,	App.IncomeUsedForCalculations				
				,	App.CurrentStepName							
				,	App.ParentStepGroupingDescription		
				,	App.CommunicationChannel					
				,	App.UserCreatedChannel										
				,	App.UserCreated
				,	ch.Channel
				,	App.SoMApplicationActivationDate
				,	App.SoMImplementationDate
				,	App.ApplicationStatusCategory
				,	CASE WHEN App.DeclaredAffordability < 0			THEN 'Negative'
						 WHEN App.DeclaredAffordability = 0			THEN '0'
						 WHEN App.DeclaredAffordability <= 50		THEN '0-50'
						 WHEN App.DeclaredAffordability <= 100		THEN '50-100'
						 WHEN App.DeclaredAffordability <= 200		THEN '100-200'
						 WHEN App.DeclaredAffordability <= 300		THEN '200-300'
						 WHEN App.DeclaredAffordability <= 400		THEN '300-400' 
						 WHEN App.DeclaredAffordability <= 500		THEN '400-500'		
						 WHEN App.DeclaredAffordability <= 600		THEN '500-600'
						 WHEN App.DeclaredAffordability <= 700		THEN '600-700'
						 WHEN App.DeclaredAffordability <= 800		THEN '700-800'
						 WHEN App.DeclaredAffordability <= 900		THEN '800-900'
						 WHEN App.DeclaredAffordability <= 1000		THEN '900-1000'
						 WHEN App.DeclaredAffordability <= 1200		THEN '1000-1200'
						 WHEN App.DeclaredAffordability <= 1400		THEN '1200-1400'
																		ELSE '1400+'
					END																			AS DeclaredAffordabilityGroup
				,	CASE WHEN App.StatedAffordability < 0			THEN 'Negative'
						 WHEN App.StatedAffordability = 0			THEN '0'
						 WHEN App.StatedAffordability <= 50			THEN '0-50'
						 WHEN App.StatedAffordability <= 100		THEN '50-100'
						 WHEN App.StatedAffordability <= 200		THEN '100-200'
						 WHEN App.StatedAffordability <= 300		THEN '200-300'
						 WHEN App.StatedAffordability <= 400		THEN '300-400'
						 WHEN App.StatedAffordability <= 500		THEN '400-500'		
						 WHEN App.StatedAffordability <= 600		THEN '500-600'
						 WHEN App.StatedAffordability <= 700		THEN '600-700'
						 WHEN App.StatedAffordability <= 800		THEN '700-800'
						 WHEN App.StatedAffordability <= 900		THEN '800-900'
						 WHEN App.StatedAffordability <= 1000		THEN '900-1000'
						 WHEN App.StatedAffordability <= 1200		THEN '1000-1200'
						 WHEN App.StatedAffordability <= 1400		THEN '1200-1400'
																		ELSE '1400+'
					END																			AS StatedAffordabilityGroup
				,	CASE WHEN App.IncomeUsedForCalculations < 0			THEN 'Negative'
						 WHEN App.IncomeUsedForCalculations = 0			THEN '0'
						 WHEN App.IncomeUsedForCalculations <= 50		THEN '0-50'
						 WHEN App.IncomeUsedForCalculations <= 100		THEN '50-100'
						 WHEN App.IncomeUsedForCalculations <= 200		THEN '100-200'
						 WHEN App.IncomeUsedForCalculations <= 300		THEN '200-300'
						 WHEN App.IncomeUsedForCalculations <= 400		THEN '300-400'
						 WHEN App.IncomeUsedForCalculations <= 500		THEN '400-500'		
						 WHEN App.IncomeUsedForCalculations <= 600		THEN '500-600'
						 WHEN App.IncomeUsedForCalculations <= 700		THEN '600-700'
						 WHEN App.IncomeUsedForCalculations <= 800		THEN '700-800'
						 WHEN App.IncomeUsedForCalculations <= 900		THEN '800-900'
						 WHEN App.IncomeUsedForCalculations <= 1000		THEN '900-1000'
						 WHEN App.IncomeUsedForCalculations <= 1200		THEN '1000-1200'
						 WHEN App.IncomeUsedForCalculations <= 1400		THEN '1200-1400'
																			ELSE '1400+'
					END																			AS IncomeUsedForCalculationsGroup
				,	CASE WHEN App.IncomeUsedForCalculations = App.StatedAffordability AND 
							  App.IncomeUsedForCalculations = App.DeclaredAffordability			THEN 'Declared/Stated'
						 WHEN App.IncomeUsedForCalculations = App.StatedAffordability			THEN 'Stated'
						 WHEN App.IncomeUsedForCalculations = App.DeclaredAffordability			THEN 'Declared'
																										ELSE 'FLDT'
					END																			AS FLDTIncomeSource
				,	SoMApp.ArrearsStatusENG AS SoMAppDateArrearsStatusENG
				,	SoMApp.SourceBucket AS SoMAppDateSourceBucket
				,	SoMApp.GBV AS SoMAppDateGBVEUR
				,	SoMApp.OnBalance AS SoMAppDateOnBalanceAmountEUR
				,	SoMApp.ObjectiveStatusDate AS SoMAppDateObjectiveDate
				,	SoMApp.ObjectiveStatus AS SoMAppDateObjectiveStatus
				,	IF(app.ApplicationStatusCategory = 'Implemented',  coalesce(SoMImp.ArrearsStatusENG, accPrv.ArrearsStatusENG),  NULL) AS SoMImplDateArrearsStatusENG
				,	IF(app.ApplicationStatusCategory = 'Implemented', coalesce(SoMImp.SourceBucket, accPrv.SourceBucket), NULL)	AS SoMImplDateSourceBucket
				,	IF(app.ApplicationStatusCategory = 'Implemented', SoMImp.GBV, NULL)	AS SoMImplDateGBVEur
				,	IF(app.ApplicationStatusCategory = 'Implemented', SoMImp.OnBalance, NULL) AS SoMImplDateOnBalanceAmountEur
				,	IF(app.ApplicationStatusCategory = 'Implemented',coalesce(SoMImp.ObjectiveStatusDate, accPrv.ObjectiveStatusDate), NULL) AS SoMImplDateObjectiveDate
				,	IF(app.ApplicationStatusCategory = 'Implemented', SoMImp.ObjectiveStatus, NULL) AS SoMImplSoMDateObjectiveStatus
				,	IF(app.ApplicationStatusCategory = 'Implemented', accNew.ArrearsStatusENG, accPrv.ArrearsStatusENG)	AS ArrearsStatusENG				
				,	IF(app.ApplicationStatusCategory = 'Implemented', accNew.SourceBucket, accPrv.SourceBucket)	AS SourceBucket			
				,	IF(app.ApplicationStatusCategory = 'Implemented', accNew.GBV, accPrv.GBV) AS GBVEur				
				,	IF(app.ApplicationStatusCategory = 'Implemented', accNew.OnBalance, accPrv.OnBalance) AS OnBlanceAmountEur				
				,	IF(app.ApplicationStatusCategory = 'Implemented', accNew.DueAmount, accPrv.DueAmount) AS DelinquentEur			
				,	IF(app.ApplicationStatusCategory = 'Implemented', accNew.GBVAtModificationDate, NULL) AS GBVAfterModificationEur
				,	IF(app.ApplicationStatusCategory = 'Implemented', pay.PaymentAmountEUR, NULL) AS ImplementedTotalPaymentsEUR	
				,	PS.InstallmentAmount AS ImplementedInstallmentAmount
				,	CASE WHEN app.ApplicationStatusCategory <> 'Implemented'	THEN NULL
						 WHEN  PS.InstallmentAmount <= 50						THEN '0-50'
						 WHEN  PS.InstallmentAmount <= 100						THEN '50-100'
						 WHEN  PS.InstallmentAmount <= 150						THEN '100-150'
						 WHEN  PS.InstallmentAmount <= 200						THEN '150-200'
						 WHEN  PS.InstallmentAmount <= 300						THEN '200-300'
						 WHEN  PS.InstallmentAmount <= 400						THEN '300-400' 
						 WHEN  PS.InstallmentAmount <= 500						THEN '400-500'		
																					ELSE '500+'
					END AS ImplementedInstallmentamountGroup 
				,	IF(app.ApplicationStatusCategory = 'Implemented' AND IFNULL(PS.InstallmentAmount,0)>0, 
						CEILING(accNew.DueAmount / PS.InstallmentAmount), NULL) AS ImplementedDueInstallmentBucket
				,	IF(app.ApplicationStatusCategory = 'Implemented', DATEDIFF(day, app.ApplicationActivationDate ,app.ImplementationDate), NULL) AS DaysToImplementation
				,	IF(app.ApplicationStatusCategory = 'Canceled', DATEDIFF(day, app.ApplicationActivationDate ,app.ApplicationCancelledDate), NULL) AS DaysToCancellation
				,	IF(app.ApplicationStatusCategory = 'Rejected', DATEDIFF(day, app.ApplicationActivationDate ,app.ApplicationRejectedDate), NULL)	AS DaysToRejection
				,	IF(app.ApplicationStatusCategory = 'Active', DATEDIFF(day, app.ApplicationActivationDate , {run_date!r}), NULL)	AS DaysActive
				,	CASE WHEN IF(app.ApplicationStatusCategory = 'Implemented', pay.PaymentAmountEUR, NULL) IS NULL					 THEN NULL
						 WHEN IF(app.ApplicationStatusCategory = 'Implemented', pay.PaymentAmountEUR, NULL) >= PS.InstallmentAmount THEN 'PaidMoreThanOneInstallment' 
																																	ELSE 'PaidLessThanOneInstallment'
					END	AS ImplementedPaymentcategory 
		FROM Applications							 App 
		LEFT JOIN Account					  AS SoMApp	ON SoMApp.ExposureKey = App.ExposureKey			AND SoMApp.BusinessDate = App.SoMApplicationActivationDate
		LEFT JOIN Account					  AS SoMImp	ON SoMImp.ExposureKey = App.ExposureKey			AND SoMImp.BusinessDate = App.SoMImplementationDate	
		LEFT JOIN Account					  AS accNew	ON accNew.ExposureKey = App.NewAccountNumber	AND accNew.BusinessDate = {run_date!r}
		LEFT JOIN Account					  AS accPrv	ON accPrv.ExposureKey = App.Exposurekey			AND accPrv.BusinessDate = {run_date!r}
		LEFT JOIN PaymentsAgg				  AS pay		ON app.NewAccountNumber = pay.ExposureKey		
		LEFT JOIN FirstInstallment		AS PS		ON PS.ExposureKey = App.NewAccountNumber
		LEFT JOIN VW_FLDTChannel		  AS ch		ON ch.UserCreatedChannel = App.UserCreatedChannel
'''
)
ModificationsMonitoring.createOrReplaceTempView("ModificationsMonitoring")

#### final_df_str

In [0]:
final_df = spark.sql(
f'''
SELECT 
    CAST({run_date!r} AS DATE) AS BusinessDate,
    CAST(PortfolioName AS STRING) AS PortfolioName,
    CAST(CDI AS STRING) AS CDI,
    CAST(AgreementKey AS STRING) AS AgreementKey,
    CAST(ExposureKey AS STRING) AS ExposureKey,
    CAST(AssetSegment AS STRING) AS AssetSegment,
    CAST(IsActiveFlag AS BOOLEAN) AS IsActiveFlag,
    CAST(ApplicationKey AS STRING) AS ApplicationKey,
    CAST(ApplicationDateCreated AS DATE) AS ApplicationDateCreated,
    CAST(ApplicationDateUpdated AS DATE) AS ApplicationDateUpdated,
    CAST(ApplicationActivationDate AS DATE) AS ApplicationActivationDate,
    CAST(ApplicationStatus AS STRING) AS ApplicationStatus,
    CAST(ApplicationCancelledDate AS DATE) AS ApplicationCancelledDate,
    CAST(ApplicationRejectedDate AS DATE) AS ApplicationRejectedDate,
    CAST(AccountImplementationStatus AS STRING) AS AccountImplementationStatus,
    CAST(ImplementationDate AS DATE) AS ImplementationDate,
    CAST(NewAccountNumber AS STRING) AS NewExposureKey,
    CAST(ModificationType AS STRING) AS ModificationType,
    CAST(CreditType AS STRING) AS CreditType,
    CAST(FlowSegment AS STRING) AS FlowSegment,
    CAST(DeclaredAffordability AS DECIMAL(19,2)) AS DeclaredAffordability,
    CAST(StatedAffordability AS DECIMAL(19,2)) AS StatedAffordability,
    CAST(IncomeUsedForCalculations AS DECIMAL(19,2)) AS IncomeUsedForCalculations,
    CAST(CurrentStepName AS STRING) AS CurrentStepName,
    CAST(ParentStepGroupingDescription AS STRING) AS ParentStepGroupingDescription,
    CAST(CommunicationChannel AS STRING) AS CommunicationChannel,
    CAST(UserCreatedChannel AS STRING) AS UserCreatedChannel,
    CAST(UserCreated AS STRING) AS UserCreated,
    CAST(Channel AS STRING) AS Channel,
    CAST(SoMApplicationActivationDate AS DATE) AS SoMApplicationActivationDate,
    CAST(SoMImplementationDate AS DATE) AS SoMImplementationDate,
    CAST(ApplicationStatusCategory AS STRING) AS ApplicationStatusCategory,
    CAST(DeclaredAffordabilityGroup AS STRING) AS DeclaredAffordabilityGroup,
    CAST(StatedAffordabilityGroup AS STRING) AS StatedAffordabilityGroup,
    CAST(IncomeUsedForCalculationsGroup AS STRING) AS IncomeUsedForCalculationsGroup,
    CAST(FLDTIncomeSource AS STRING) AS FLDTIncomeSource,
    CAST(SoMAppDateArrearsStatusENG AS STRING) AS SoMAppDateArrearsStatusENG,
    CAST(SoMAppDateSourceBucket AS INT) AS SoMAppDateSourceBucket,
    CAST(SoMAppDateGBVEUR AS DECIMAL(19,2)) AS SoMAppDateGBVEUR,
    CAST(SoMAppDateOnBalanceAmountEUR AS DECIMAL(19,2)) AS SoMAppDateOnBalanceAmountEUR,
    CAST(SoMAppDateObjectiveDate AS DATE) AS SoMAppDateObjectiveDate,
    CAST(SoMAppDateObjectiveStatus AS STRING) AS SoMAppDateObjectiveStatus,
    CAST(SoMImplDateArrearsStatusENG AS STRING) AS SoMImplDateArrearsStatusENG,
    CAST(SoMImplDateSourceBucket AS INT) AS SoMImplDateSourceBucket,
    CAST(SoMImplDateGBVEur AS DECIMAL(19,2)) AS SoMImplDateGBVEur,
    CAST(SoMImplDateOnBalanceAmountEur AS DECIMAL(19,2)) AS SoMImplDateOnBalanceAmountEur,
    CAST(SoMImplDateObjectiveDate AS DATE) AS SoMImplDateObjectiveDate,
    CAST(SoMImplSoMDateObjectiveStatus AS STRING) AS SoMImplSoMDateObjectiveStatus,
    CAST(ArrearsStatusENG AS STRING) AS ArrearsStatusENG,
    CAST(SourceBucket AS INT) AS SourceBucket,
    CAST(GBVEur AS DECIMAL(19,2)) AS GBVEur,
    CAST(OnBlanceAmountEur AS DECIMAL(19,2)) AS OnBlanceAmountEur,
    CAST(DelinquentEur AS DECIMAL(19,2)) AS DelinquentEur,
    CAST(GBVAfterModificationEur AS DECIMAL(19,2)) AS GBVAfterModificationEur,
    CAST(ImplementedTotalPaymentsEUR AS DECIMAL(19,2)) AS ImplementedTotalPaymentsEUR,
    CAST(ImplementedInstallmentAmount AS DECIMAL(19,2)) AS ImplementedInstallmentAmount,
    CAST(ImplementedInstallmentamountGroup AS STRING) AS ImplementedInstallmentamountGroup,
    CAST(ImplementedDueInstallmentBucket AS INT) AS ImplementedDueInstallmentBucket,
    CAST(DaysToImplementation AS INT) AS DaysToImplementation,
    CAST(DaysToCancellation AS INT) AS DaysToCancellation,
    CAST(DaysToRejection AS INT) AS DaysToRejection,
    CAST(DaysActive AS INT) AS DaysActive,
    CAST(ImplementedPaymentcategory AS STRING) AS ImplementedPaymentcategory,
    CAST(getdate() AS TIMESTAMP) AS InsertTimeStamp
FROM ModificationsMonitoring;
'''
)
final_df.createOrReplaceTempView("final_df")


In [0]:
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[coalesce(col(c), lit("''")) for c in key_columns]).alias("CompositeKey")
            # composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["ExposureKey", "NewExposureKey"]
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
ModificationsMonitoring = spark.sql(
f'''
select * from {delivery_table_path}
where BusinessDate = {run_date!r}
'''
)
ModificationsMonitoring.createOrReplaceTempView("ModificationsMonitoring")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/HAPS/ModificationsMonitoring/" # 
file_name = "ModificationsMonitoring_" +run_date+ ".csv"
ModificationsMonitoring.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}") # 
rename_and_cleanup_files(output_path, file_name)